In [1]:
import mne
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# MNE log mesajlarını azalt
mne.set_log_level('WARNING')

In [2]:
# =============================================================================
# PARAMETRE TANIMLAMA
# =============================================================================
"""
NEDEN: Tüm ön işleme parametrelerini merkezi bir yerde tanımlayarak
       kodun okunabilirliğini ve yeniden kullanılabilirliğini artırırız.
       
NASIL: Veri seti yolu, denek sayısı, filtre parametreleri ve epoch
       pencere değerlerini değişkenler olarak tanımlarız.
"""

# Veri seti parametreleri
DATA_DIR = '../physioNet_Dataset'  # Veri seti ana klasörü
N_SUBJECTS = 40  # Toplam denek sayısı (1-109 arası, biz 40 kullanıyoruz)

# Motor imgelem koşuları içeren run'lar
# R03, R04: Sol vs Sağ el (göze açık)
# R07, R08: Yumruk vs Ayak (göze açık)  
# R11, R12: Sol vs Sağ el (göze kapalı)
IMAGERY_RUNS = ['R03', 'R04', 'R07', 'R08', 'R11', 'R12']

# Frekans filtreleme parametreleri
L_FREQ = 0.5   # Alt kesim frekansı (Hz) - Düşük frekans gürültüsünü temizler
H_FREQ = 40.0  # Üst kesim frekansı (Hz) - Yüksek frekans gürültüsünü temizler

# Epoch pencereleme parametreleri
TMIN = 1.0  # Epoch başlangıcı (stimulus sonrası 1. saniye)
TMAX = 4.0  # Epoch sonu (stimulus sonrası 4. saniye)

print("=" * 70)
print("VERİ ÖN İŞLEME PARAMETRELERİ")
print("=" * 70)
print(f"Denek Sayısı      : {N_SUBJECTS}")
print(f"Koşu/Denek        : {len(IMAGERY_RUNS)} → ~{len(IMAGERY_RUNS)*30} epoch/denek")
print(f"Filtre Aralığı    : {L_FREQ}-{H_FREQ} Hz (geniş bant)")
print(f"Epoch Penceresi   : {TMIN}-{TMAX} saniye")
print("=" * 70)
print()


VERİ ÖN İŞLEME PARAMETRELERİ
Denek Sayısı      : 40
Koşu/Denek        : 6 → ~180 epoch/denek
Filtre Aralığı    : 0.5-40.0 Hz (geniş bant)
Epoch Penceresi   : 1.0-4.0 saniye



In [3]:
# =============================================================================
# 1. HAM EEG VERİSİNİ YÜKLEME
# =============================================================================
"""
NEDEN: EDF (European Data Format) dosyaları ham EEG verilerini içerir.
       Bu verileri Python ortamına yüklemek için MNE kütüphanesi kullanılır.
       
NASIL: mne.io.read_raw_edf() fonksiyonu ile EDF dosyası okunur.
       preload=True parametresi tüm veriyi belleğe yükler (hızlı işleme için).
"""

def adim1_ham_veri_yukle(subject_id, runs, data_dir):
    """
    Bir deneğin belirtilen koşularındaki ham EEG verilerini yükler.
    
    Parametreler:
    ------------
    subject_id : int
        Denek numarası (1-109 arası)
    runs : list
        Yüklenecek koşu kodları (örn: ['R03', 'R04'])
    data_dir : str
        Veri seti ana klasör yolu
        
    Dönüş:
    ------
    list : MNE Raw nesnelerinin listesi
    """
    print(f"\n[ADIM 1] Ham EEG Verisi Yükleme - S{subject_id:03d}")
    print("-" * 60)
    
    # Denek klasör adını oluştur (örn: S001, S002, ...)
    sub = f'S{subject_id:03d}'
    raws = []
    
    # Her bir koşu için EDF dosyasını yükle
    for run in runs:
        # Dosya yolu: örnek -> ../physioNet_Dataset/S001/S001R03.edf
        file_path = f'{data_dir}/{sub}/{sub}{run}.edf'
        
        try:
            # EDF dosyasını oku
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
            raws.append(raw)
            print(f"  ✓ {run} yüklendi: {len(raw.times)} zaman noktası, "
                  f"{len(raw.ch_names)} kanal")
            
        except FileNotFoundError:
            print(f"  ✗ {run} bulunamadı!")
            
    print(f"  TOPLAM: {len(raws)} koşu yüklendi")
    
    return raws


In [4]:
# =============================================================================
# 2. VERİLERİ BİRLEŞTİRME
# =============================================================================
"""
NEDEN: Her koşu ayrı bir dosyada olduğu için bunları birleştirmek gerekir.
       Bu sayede tüm veri tek bir Raw nesnesinde toplanır ve işlemler kolaylaşır.
       
NASIL: mne.concatenate_raws() fonksiyonu birden fazla Raw nesnesini birleştirir.
       Zaman ekseni boyunca ardışık olarak eklenir.
"""

def adim2_verileri_birlestir(raws):
    """
    Birden fazla Raw nesnesini tek bir Raw nesnesinde birleştirir.
    
    Parametreler:
    ------------
    raws : list
        MNE Raw nesnelerinin listesi
        
    Dönüş:
    ------
    mne.io.Raw : Birleştirilmiş Raw nesnesi
    """
    print("\n[ADIM 2] Verileri Birleştirme")
    print("-" * 60)
    
    if not raws:
        print("  ✗ Birleştirilecek veri yok!")
        return None
    
    # Raw nesnelerini birleştir
    raw_combined = mne.concatenate_raws(raws)
    
    print(f"  ✓ {len(raws)} koşu birleştirildi")
    print(f"  ✓ Toplam süre: {raw_combined.times[-1]:.1f} saniye")
    print(f"  ✓ Örnekleme frekansı: {raw_combined.info['sfreq']} Hz")
    
    return raw_combined


In [5]:
# =============================================================================
# 3. KANAL İSİMLERİNİ STANDARTLAŞTIRMA
# =============================================================================
"""
NEDEN: EDF dosyalarındaki kanal isimleri tutarsız olabilir (örn: 'Fc5.' veya 'FC5').
       Standart 10-20 sistemine uygun hale getirmek gerekir.
       
NASIL: Kanal isimlerindeki nokta karakterleri kaldırılır ve büyük harfe çevrilir.
       Sonra MNE'nin standart montaj sistemi uygulanır.
"""

def adim3_kanal_isimleri_standartlastir(raw):
    """
    Kanal isimlerini temizler ve standart 10-20 montajını uygular.
    
    Parametreler:
    ------------
    raw : mne.io.Raw
        Ham EEG verisi
        
    Dönüş:
    ------
    mne.io.Raw : Güncellenmiş Raw nesnesi
    """
    print("\n[ADIM 3] Kanal İsimleri Standartlaştırma")
    print("-" * 60)
    
    # Eski kanal isimlerini kaydet
    old_names = raw.ch_names[:5]  # İlk 5 kanal
    
    # Kanal isimlerini temizle: nokta kaldır ve büyük harfe çevir
    # Örnek: 'Fc5.' -> 'FC5'
    raw.rename_channels({ch: ch.rstrip('.').upper() for ch in raw.ch_names})
    
    # Standart 10-20 elektrot pozisyon sistemini uygula
    # Bu, her kanalın kafadaki konumunu tanımlar
    raw.set_montage(
        mne.channels.make_standard_montage('standard_1020'),
        on_missing='ignore'  # Tanımlı olmayan kanalları atla
    )
    
    print(f"  ✓ Önceki isimler: {old_names[:3]}")
    print(f"  ✓ Yeni isimler  : {raw.ch_names[:3]}")
    print(f"  ✓ Montaj uygulandı: 'standard_1020'")
    
    return raw

In [6]:
# =============================================================================
# 4. SADECE EEG KANALLARINI SEÇME
# =============================================================================
"""
NEDEN: Raw veri EEG dışında başka kanal tipleri de içerebilir (EOG, EMG, vb.).
       Sadece EEG kanallarıyla çalışmak istiyoruz.
       
NASIL: pick() fonksiyonu belirtilen kanal tipini seçer ve diğerlerini atar.
"""

def adim4_eeg_kanallari_sec(raw):
    """
    Sadece EEG kanallarını seçer, diğer kanal tiplerini çıkarır.
    
    Parametreler:
    ------------
    raw : mne.io.Raw
        Ham EEG verisi
        
    Dönüş:
    ------
    mne.io.Raw : Sadece EEG kanallarını içeren Raw nesnesi
    """
    print("\n[ADIM 4] Sadece EEG Kanallarını Seçme")
    print("-" * 60)
    
    # Filtreleme öncesi kanal sayısı
    n_channels_before = len(raw.ch_names)
    
    # Sadece 'eeg' tipindeki kanalları seç
    raw.pick('eeg')
    
    # Filtreleme sonrası kanal sayısı
    n_channels_after = len(raw.ch_names)
    
    print(f"  ✓ Toplam kanal  : {n_channels_before}")
    print(f"  ✓ EEG kanalları : {n_channels_after}")
    print(f"  ✓ Çıkarılan     : {n_channels_before - n_channels_after}")
    
    return raw


In [7]:
# =============================================================================
# 5. BANT GEÇİREN FİLTRE UYGULAMA (BAND-PASS FILTER)
# =============================================================================
"""
NEDEN: Ham EEG sinyali geniş frekans aralığında gürültü içerir:
       - Çok düşük frekanslarda (<0.5 Hz): Yavaş kayma (drift)
       - Çok yüksek frekanslarda (>40 Hz): Kas aktivitesi (EMG gürültüsü)
       Motor imgelem için önemli olan 8-30 Hz (mu ve beta bandları) aralığıdır.
       Ancak CNN bu bantları kendisi öğreneceği için geniş bant (0.5-40 Hz) kullanırız.
       
NASIL: Finite Impulse Response (FIR) filtre ile belirtilen frekans aralığı dışındaki
       sinyaller zayıflatılır. 'firwin' pencere fonksiyonu kullanılır.
"""

def adim5_bant_geciren_filtre(raw, l_freq, h_freq):
    """
    Bant geçiren (band-pass) filtre uygular.
    
    Parametreler:
    ------------
    raw : mne.io.Raw
        Ham EEG verisi
    l_freq : float
        Alt kesim frekansı (Hz)
    h_freq : float
        Üst kesim frekansı (Hz)
        
    Dönüş:
    ------
    mne.io.Raw : Filtrelenmiş Raw nesnesi
    """
    print("\n[ADIM 5] Bant Geçiren Filtre Uygulama")
    print("-" * 60)
    
    print(f"  Filtre aralığı: {l_freq} - {h_freq} Hz")
    print("  Filtre tipi   : FIR (Finite Impulse Response)")
    print("  Pencere       : Hamming (firwin)")
    
    # Bant geçiren filtre uygula
    raw.filter(
        l_freq,           # Alt kesim frekansı
        h_freq,           # Üst kesim frekansı
        fir_design='firwin',  # FIR filtre tasarımı
        verbose=False     # Detaylı çıktı kapalı
    )
    
    print("  ✓ Filtreleme tamamlandı")
    print(f"  ✓ 0-{l_freq} Hz ve {h_freq}+ Hz zayıflatıldı")
    
    return raw

In [8]:
# =============================================================================
# 6. NOTCH FİLTRESİ (ÇENTİK FİLTRESİ)
# =============================================================================
"""
NEDEN: Elektrik şebekesi 50 Hz (Avrupa) veya 60 Hz (Amerika) frekansta sinüzoidal
       gürültü yaratır. Bu gürültü EEG sinyalini kirletir ve analizi zorlaştırır.
       
NASIL: Belirli bir frekanstaki (50 veya 60 Hz) sinyali bastıran dar bantlı
       bir filtre (notch filter) kullanılır.
"""

def adim6_notch_filtresi(raw, notch_freq=60):
    """
    Şebeke gürültüsünü temizlemek için notch (çentik) filtresi uygular.
    
    Parametreler:
    ------------
    raw : mne.io.Raw
        Ham EEG verisi
    notch_freq : int
        Bastırılacak frekans (50 Hz Avrupa, 60 Hz Amerika)
        
    Dönüş:
    ------
    mne.io.Raw : Notch filtreli Raw nesnesi
    """
    print("\n[ADIM 6] Notch Filtresi (Şebeke Gürültüsü Temizleme)")
    print("-" * 60)
    
    print(f"  Hedef frekans : {notch_freq} Hz (şebeke gürültüsü)")
    print("  Amaç          : Elektrik şebekesinden gelen sinüzoidal gürültüyü bastırma")
    
    # Notch filtre uygula
    raw.notch_filter(notch_freq, verbose=False)
    
    print(f"  ✓ {notch_freq} Hz notch filtre uygulandı")
    
    return raw

In [9]:
# =============================================================================
# 7. EVENT (OLAY) ÇIKARMA
# =============================================================================
"""
NEDEN: EEG kaydı sürekli bir sinyal akışıdır. Hangi zaman aralıklarının
       hangi görevle ilişkili olduğunu bilmemiz gerekir. EDF dosyasındaki
       annotation'lar (T0, T1, T2) bu bilgiyi içerir.
       
NASIL: mne.events_from_annotations() fonksiyonu annotation'ları olay
       zamanlarına (events) dönüştürür. Her olay [zaman, 0, kod] şeklindedir.
"""

def adim7_event_cikarma(raw):
    """
    Annotation'lardan event (olay) bilgilerini çıkarır.
    
    Event kodları:
    - T0: Baseline (dinlenme)
    - T1: Sol el motor imgelem
    - T2: Sağ el motor imgelem
    
    Parametreler:
    ------------
    raw : mne.io.Raw
        Ham EEG verisi
        
    Dönüş:
    ------
    events : ndarray, shape (n_events, 3)
        Her satır [zaman_örneği, 0, olay_kodu]
    event_id : dict
        Olay ismi -> kod eşleşmesi
    """
    print("\n[ADIM 7] Event (Olay) Çıkarma")
    print("-" * 60)
    
    # Annotation'lardan event'leri çıkar
    events, event_dict = mne.events_from_annotations(raw, verbose=False)
    
    print(f"  ✓ Toplam {len(events)} event bulundu")
    print("  Event türleri:")
    for event_name, event_code in event_dict.items():
        count = np.sum(events[:, 2] == event_code)
        print(f"    - {event_name:4s} (kod {event_code}): {count} kez")
    
    # Sadece T1 (sol el) ve T2 (sağ el) event'lerini filtrele
    # T0 (baseline) ve diğer event'ler çıkarılır
    event_id_filtered = {k: v for k, v in event_dict.items() if k in ['T1', 'T2']}
    
    if not event_id_filtered:
        print("  ✗ T1 veya T2 event'i bulunamadı!")
        return None, None
    
    print(f"\n  Filtrelenmiş event'ler (sadece T1 ve T2):")
    for name, code in event_id_filtered.items():
        count = np.sum(events[:, 2] == code)
        print(f"    - {name}: {count} epoch")
    
    return events, event_id_filtered


In [10]:
# =============================================================================
# 8. EPOCH OLUŞTURMA
# =============================================================================
"""
NEDEN: Sürekli EEG sinyalini, her bir görev için belirli zaman pencerelerine
       (epoch) bölmek gerekir. Her epoch bir denemeyi temsil eder.
       
NASIL: mne.Epochs() fonksiyonu event zamanlarını kullanarak sinyali böler.
       tmin ve tmax parametreleri epoch penceresini belirler.
       baseline=None: Motor imgelem çalışmalarında baseline düzeltmesi yapılmaz,
                      çünkü tüm pencere aktif görev içerir.
"""

def adim8_epoch_olusturma(raw, events, event_id, tmin, tmax):
    """
    Event'lere göre epoch'ları (denemeleri) oluşturur.
    
    Parametreler:
    ------------
    raw : mne.io.Raw
        Filtrelenmiş EEG verisi
    events : ndarray
        Event zamanları ve kodları
    event_id : dict
        Event isim -> kod eşleşmesi
    tmin : float
        Epoch başlangıcı (event'e göre saniye)
    tmax : float
        Epoch sonu (event'e göre saniye)
        
    Dönüş:
    ------
    epochs : mne.Epochs
        Epoch'ların koleksiyonu
    """
    print("\n[ADIM 8] Epoch Oluşturma")
    print("-" * 60)
    
    print(f"  Epoch penceresi   : {tmin} - {tmax} saniye (görev sonrası)")
    print(f"  Baseline düzeltme : Yok (motor imgelem için uygun değil)")
    print(f"  Projektör         : Uygulanıyor (artifact temizleme için)")
    
    # Epoch'ları oluştur
    epochs = mne.Epochs(
        raw,
        events,           # Event zamanları
        event_id,         # Hangi event'leri kullanacağız
        tmin=tmin,        # Epoch başlangıcı
        tmax=tmax,        # Epoch sonu
        proj=True,        # SSP projektörlerini uygula
        picks='eeg',      # Sadece EEG kanalları
        baseline=None,    # Baseline düzeltme yok
        preload=True,     # Veriyi belleğe yükle
        verbose=False
    )
    
    print(f"  ✓ {len(epochs)} epoch oluşturuldu")
    print(f"  ✓ Epoch boyutu: {epochs.get_data().shape}")
    print(f"    - {epochs.get_data().shape[0]} epoch")
    print(f"    - {epochs.get_data().shape[1]} kanal")
    print(f"    - {epochs.get_data().shape[2]} zaman noktası")
    
    return epochs

In [11]:
# =============================================================================
# 9. KÖTÜ EPOCH'LARI ÇIKARMA
# =============================================================================
"""
NEDEN: Bazı epoch'lar çok fazla artifact (göz kırpma, hareket, vb.) içerebilir.
       Bu epoch'lar modeli yanıltabilir ve performansı düşürür.
       
NASIL: MNE otomatik olarak amplitüd eşiğini aşan epoch'ları işaretler.
       drop_bad() fonksiyonu bu işaretli epoch'ları çıkarır.
"""

def adim9_kotu_epochlari_cikar(epochs):
    """
    Aşırı artifact içeren kötü epoch'ları tespit edip çıkarır.
    
    Parametreler:
    ------------
    epochs : mne.Epochs
        Epoch koleksiyonu
        
    Dönüş:
    ------
    epochs : mne.Epochs
        Temizlenmiş epoch koleksiyonu
    """
    print("\n[ADIM 9] Kötü Epoch'ları Çıkarma")
    print("-" * 60)
    
    n_before = len(epochs)
    
    # Kötü epoch'ları otomatik tespit et ve çıkar
    # Eşik değeri: Peak-to-peak amplitüd 100 µV'u aşarsa kötü kabul edilir
    epochs.drop_bad(verbose=False)
    
    n_after = len(epochs)
    n_dropped = n_before - n_after
    
    print(f"  ✓ Başlangıç epoch : {n_before}")
    print(f"  ✓ Çıkarılan epoch : {n_dropped} (%{100*n_dropped/n_before:.1f})")
    print(f"  ✓ Kalan epoch     : {n_after}")
    
    return epochs

In [12]:
# =============================================================================
# 10. VERİYİ NUMPY ARRAY'E DÖNÜŞTÜRME
# =============================================================================
"""
NEDEN: MNE Epochs nesnesi derin öğrenme modelleri için uygun değildir.
       NumPy array formatına çevirmek gerekir.
       
NASIL: get_data() metodu epoch'ları (n_epochs, n_channels, n_times) şeklinde
       NumPy array'e dönüştürür. float32 tipi bellek tasarrufu sağlar.
"""

def adim10_numpy_array_donustur(epochs):
    """
    Epoch'ları NumPy array formatına dönüştürür.
    
    Parametreler:
    ------------
    epochs : mne.Epochs
        Epoch koleksiyonu
        
    Dönüş:
    ------
    X : ndarray, shape (n_epochs, n_channels, n_times)
        EEG sinyalleri
    """
    print("\n[ADIM 10] NumPy Array'e Dönüştürme")
    print("-" * 60)
    
    # Epoch'ları NumPy array'e çevir
    X = epochs.get_data().astype(np.float32)
    
    print(f"  ✓ Dönüşüm tamamlandı")
    print(f"  ✓ Array boyutu: {X.shape}")
    print(f"    - Veri tipi : {X.dtype}")
    print(f"    - Bellek    : {X.nbytes / 1024 / 1024:.2f} MB")
    
    return X

In [13]:
# =============================================================================
# 11. ETİKET KODLAMA (LABEL ENCODING)
# =============================================================================
"""
NEDEN: Event kodları (örn: T1=2, T2=3) sürekli değildir. Sınıflandırma
       modelleri için 0'dan başlayan sürekli sınıf etiketleri gerekir.
       
NASIL: sklearn'ın LabelEncoder'ı string/int etiketleri 0, 1, 2, ... şeklinde
       kodlar. Örnek: T1 -> 0, T2 -> 1
"""

def adim11_etiket_kodlama(epochs):
    """
    Epoch etiketlerini (T1, T2) sayısal sınıf etiketlerine (0, 1) dönüştürür.
    
    Parametreler:
    ------------
    epochs : mne.Epochs
        Epoch koleksiyonu
        
    Dönüş:
    ------
    y : ndarray, shape (n_epochs,)
        Kodlanmış sınıf etiketleri (0: T1, 1: T2)
    """
    print("\n[ADIM 11] Etiket Kodlama")
    print("-" * 60)
    
    # Event kodlarını al (epoch'ların 3. sütunu)
    event_codes = epochs.events[:, -1]
    
    # Label encoder oluştur ve fit et
    le = LabelEncoder()
    y = le.fit_transform(event_codes).astype(np.int64)
    
    print(f"  ✓ Orijinal kodlar   : {np.unique(event_codes)}")
    print(f"  ✓ Dönüştürülmüş     : {np.unique(y)}")
    print(f"  ✓ Etiket dağılımı   :")
    
    for label_idx, label_name in enumerate(le.classes_):
        count = np.sum(y == label_idx)
        print(f"    - Sınıf {label_idx} ({label_name}): {count} epoch")
    
    return y

In [14]:
# =============================================================================
# ANA FONKSİYON - TÜM ÖN İŞLEME ADIMLARINI BİRLEŞTİR
# =============================================================================

def veri_on_isleme_pipeline(subject_id, runs, data_dir, l_freq, h_freq, tmin, tmax):
    """
    Bir denek için tüm veri ön işleme adımlarını sırayla uygular.
    
    Parametreler:
    ------------
    subject_id : int
        Denek numarası
    runs : list
        Yüklenecek koşu kodları
    data_dir : str
        Veri seti klasörü
    l_freq : float
        Alt kesim frekansı (Hz)
    h_freq : float
        Üst kesim frekansı (Hz)
    tmin : float
        Epoch başlangıcı (saniye)
    tmax : float
        Epoch sonu (saniye)
        
    Dönüş:
    ------
    X : ndarray or None
        EEG sinyalleri (n_epochs, n_channels, n_times)
    y : ndarray or None
        Sınıf etiketleri (n_epochs,)
    """
    print("\n")
    print("=" * 70)
    print(f"VERİ ÖN İŞLEME BAŞLIYOR - DENEK S{subject_id:03d}")
    print("=" * 70)
    
    # ADIM 1: Ham veri yükleme
    raws = adim1_ham_veri_yukle(subject_id, runs, data_dir)
    if not raws:
        print("\n✗ Veri yüklenemedi!")
        return None, None
    
    # ADIM 2: Verileri birleştirme
    raw = adim2_verileri_birlestir(raws)
    if raw is None:
        return None, None
    
    # ADIM 3: Kanal isimlerini standartlaştırma
    raw = adim3_kanal_isimleri_standartlastir(raw)
    
    # ADIM 4: Sadece EEG kanallarını seçme
    raw = adim4_eeg_kanallari_sec(raw)
    
    # ADIM 5: Bant geçiren filtre
    raw = adim5_bant_geciren_filtre(raw, l_freq, h_freq)
    
    # ADIM 6: Notch filtresi
    raw = adim6_notch_filtresi(raw, notch_freq=60)
    
    # ADIM 7: Event çıkarma
    events, event_id = adim7_event_cikarma(raw)
    if events is None:
        print("\n✗ Event'ler çıkarılamadı!")
        return None, None
    
    # ADIM 8: Epoch oluşturma
    epochs = adim8_epoch_olusturma(raw, events, event_id, tmin, tmax)
    
    # ADIM 9: Kötü epoch'ları çıkarma
    epochs = adim9_kotu_epochlari_cikar(epochs)
    
    # Minimum epoch kontrolü
    if len(epochs) < 20 or len(np.unique(epochs.events[:, -1])) < 2:
        print(f"\n✗ Yetersiz epoch! (Minimum 20 gerekli, {len(epochs)} bulundu)")
        return None, None
    
    # ADIM 10: NumPy array'e dönüştürme
    X = adim10_numpy_array_donustur(epochs)
    
    # ADIM 11: Etiket kodlama
    y = adim11_etiket_kodlama(epochs)
    
    print("\n" + "=" * 70)
    print("VERİ ÖN İŞLEME TAMAMLANDI!")
    print("=" * 70)
    print(f"Çıktı Boyutları:")
    print(f"  X (Sinyal): {X.shape}")
    print(f"  y (Etiket): {y.shape}")
    print("=" * 70)
    
    return X, y

In [ ]:

if __name__ == "__main__":
    print("\n" + "="*70)
    print("EEG VERİ ÖN İŞLEME - ÖRNEK ÇALIŞTIRMA")
    print("="*70)
    
    # Tek bir denek için ön işleme yap (Denek 1)
    X, y = veri_on_isleme_pipeline(
        subject_id=1,
        runs=IMAGERY_RUNS,
        data_dir=DATA_DIR,
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        tmin=TMIN,
        tmax=TMAX
    )
    
    if X is not None:
        print("\n✓ Ön işleme başarıyla tamamlandı!")
        print(f"\nVeriler derin öğrenme modeline hazır:")
        print(f"  - {X.shape[0]} epoch")
        print(f"  - {X.shape[1]} kanal")
        print(f"  - {X.shape[2]} zaman noktası")
        print(f"  - {len(np.unique(y))} sınıf")


EEG VERİ ÖN İŞLEME - ÖRNEK ÇALIŞTIRMA


VERİ ÖN İŞLEME BAŞLIYOR - DENEK S001

[ADIM 1] Ham EEG Verisi Yükleme - S001
------------------------------------------------------------
  ✓ R03 yüklendi: 20000 zaman noktası, 64 kanal
  ✓ R04 yüklendi: 20000 zaman noktası, 64 kanal
  ✓ R07 yüklendi: 20000 zaman noktası, 64 kanal
  ✓ R08 yüklendi: 20000 zaman noktası, 64 kanal
  ✓ R11 yüklendi: 20000 zaman noktası, 64 kanal
  ✓ R12 yüklendi: 20000 zaman noktası, 64 kanal
  TOPLAM: 6 koşu yüklendi

[ADIM 2] Verileri Birleştirme
------------------------------------------------------------
  ✓ 6 koşu birleştirildi
  ✓ Toplam süre: 750.0 saniye
  ✓ Örnekleme frekansı: 160.0 Hz

[ADIM 3] Kanal İsimleri Standartlaştırma
------------------------------------------------------------
  ✓ Önceki isimler: ['Fc5.', 'Fc3.', 'Fc1.']
  ✓ Yeni isimler  : ['FC5', 'FC3', 'FC1']
  ✓ Montaj uygulandı: 'standard_1020'

[ADIM 4] Sadece EEG Kanallarını Seçme
------------------------------------------------------------